# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 0
%aimport _campaign_lib, api

In [2]:
import json
from _campaign_lib import *

svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

2026-03-11 14:08:00 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-11 14:08:00 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-11 14:08:00 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [3]:
campaign_config = {
    "queries_per_eval": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": 3,                 # default: 10
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        "model": "openai/gpt-oss-120b",
        # "model": "moonshotai/kimi-k2-instruct-0905"
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        # "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        # "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "eval_queries_per_point": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

In [4]:
#@title Pipeline snapshot (full config for reproducibility)
pipeline_config_full = await show_pipeline_snapshot(svc)

2026-03-11 14:08:00 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "meta-llama/llama-4-maverick-17b-128e-instruct",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "url_fetch_multiplier": 2,
        "fallback_keywords_limit": 8,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
      

In [5]:
#@title Build pipeline params
pipeline_params = configure_pipeline(svc, campaign_config)

Active steps: ['entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']


## 2. Data

In [6]:
#@title Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [7]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline, eval_data, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline, eval_data, campaign_config, svc,
    )

2026-03-11 14:08:00 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"



BACKEND STATUS
  Session Active                 False
  Active Sessions                0
  Terms Loaded                   0
  Match Database Identifiers     109
  Match Database Aliases         599
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [8]:
#@title Candidate coverage (post-eval diagnostic)
cov_df = run_coverage_diagnostic(
    baseline_results,
    svc["store"], svc["backend_id"], svc["experiment_id"],
)

Loaded 40 eval queries
Eval runs: 45 completed runs, 4 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0.0   50.0%     6      
  scan_dff96710         scan                                      0.0   33.3%     6      
  scan_3dbc066d         scan     

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

### 3a. Smart Search

In [9]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description="TASK_DESCRIPTION", raw=True)

2026-03-11 14:08:01 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
TASK_DESCRIPTION
## Tunable Parameters (per step)
[
  {
    "name": "fuzzy_matching",
    "param_keys": [
      "fuzzy_scorer",
      "fuzzy_threshold"
    ]
  },
  {
    "name

In [10]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
)

2026-03-11 14:08:01 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling openai/gpt-oss-120b ...



2026-03-11 14:08:06 INFO     [httpx] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-11 14:08:06 WARNING  [api.services.search.scan_advisor] Scan advisor validation: pipeline_param axis 'entity_profiling_schema' not found in PipelineSchema param_keys: ['content_char_limit', 'fuzzy_scorer', 'fuzzy_threshold', 'max_sites', 'max_token_candidates', 'num_results', 'profiling_max_tokens', 'profiling_model', 'profiling_prompt', 'profiling_schema', 'profiling_temperature', 'query_prefix', 'query_suffix', 'ranking_max_tokens', 'ranking_model', 'ranking_prompt', 'ranking_sample_size', 'ranking_schema', 'ranking_temperature', 'raw_content_limit', 'relevance_weight_core']


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     controls candidate inclusion breadth
     Values: ['0.6', '0.7', '0.8']
  2. [MEDIUM] fuzzy_scorer (pipeline_param) -- step: fuzzy_matching
     affects similarity scoring method
     Values: ['ratio', 'partial_ratio', 'token_set_ratio']
  3. [HIGH] query_prefix (pipeline_param) -- step: web_search
     guides search focus toward materials
     Values: ['"material" ', '"chemical" ']
  4. [MEDIUM] query_suffix (pipeline_param) -- step: web_search
     adds context for LCA relevance
     Values: ['"LCA"', '"environmental impact"']
  5. [MEDIUM] content_char_limit (pipeline_param) -- step: web_search
     more context improves profiling
     Values: ['1200', '2000']
  6. [HIGH] profiling_prompt (pipeline_param) -- step: entity_profiling
   

In [11]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    'profiling_max_tokens': [512, 1024, 2048],
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'')
  profiling_temperatur

In [12]:
#@title Prepare scan baseline
baseline_sp = await prepare_scan_baseline(
    baseline, campaign_config,
    pipeline_params=campaign_config.get("pipeline_params"),
)

2026-03-11 14:08:09 INFO     [httpx] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  Restructured baseline fields:
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity profile JSON and a list of candidate strings, extract the entity...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify the entity_category, an...
    thinking_style: Think step by step.
    answer_format: JSON with a 'reasoning' string and a 'ranked_candidates' array (rank, candidate,...
  Search baseline: 0de6163e0c76 (render: 1355 chars)


In [13]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    baseline_sp, scan_variants, eval_data, svc.get("backend_client"),
    store=svc["store"], backend_id=svc["backend_id"],
    pipeline_schema=svc.get("pipeline_schema"),
)

2026-03-11 14:08:09 INFO     [api.services.search.coverage] build_prompt_result_index: 45 runs -> 12 unique prompts, 2987 total query results


Running sensitivity scan...

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity profile JSON and a list of candidate strings, extract the entity...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify the entity_category, an...
    thinking_style: Think step by step.
    answer_format: JSON with a 'reasoning' string and a 'ranked_candidates' array (rank, candidate,...

  Axes: 5, variants: 17, queries/variant: 984, historical results: 2987
  Estimated calls: ~16728


2026-03-11 14:08:10 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/sessions "HTTP/1.1 200 OK"


  Evaluating baseline...
        MISS  [token]  PBT-GF30 Ultradur B4300 G6/molding             -> NO_RESULT 0.7s
        MISS --/1  [token]  Hardener TP219/n/a                             -> N-olefins {RER}| N-olefins producti 0.3s
        MISS --/2  [token]  Strip EN 1652-Cu-DHP-R290-1x73Ag0,3/stamping   -> Steel Electrogalvanized {GLO} | bla 0.3s
        MISS  [token]  Strip EN 10139-DC03+C390-MB 1,0x25GK/cold for  -> NO_RESULT 0.3s
        MISS --/3  [token]  Strip EN 1652 0.8x... CuZn37 R350/stamping     -> Galvanized Copper | 99.6% Copper 0. 0.3s
        HIT   [token]  Steel 8.8/thread rolling                       -> Sheet rolling, steel {RER}| sheet r 0.3s
        MISS  [token]  Strip EN 1652-Cu-DHP-R290-1,8x35/stamping      -> NO_RESULT 0.3s
        MISS --/2  [token]  CuZn21Si3P-H110/milling                        -> Steel removed by milling, small par 0.3s
        MISS  [token]  Strip EN 13599-Cu-PHC-R290-1,0x.../stamping    -> NO_RESULT 0.3s
        MISS --/1  [token]  EN AW

2026-03-11 14:08:33 WARNING  [api.services.prompt_eval] Eval batch interrupted at query 32/984. Partial results saved via incremental writer.
2026-03-11 14:08:33 WARNING  [api.services.prompt_eval] evaluate_prompt_cached interrupted for scan_a7f0fb3f. Partial results saved to scan_a7f0fb3f.partial.jsonl — will resume on next run.



  [INTERRUPTED] Sensitivity scan
  Saved: completed evaluations saved via evaluate_prompt_cached
  Resume: re-run this cell -- cached evals will be reused


In [ ]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

### 3b. Grid Search

<details>
<summary>Skip if you used Smart Search above.</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing.

</details>

In [17]:
#@title Grid campaign overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

Grid plans (1):
  [run..] gridplan_d603214cea60  34 points  (space=96, axes=persona,task_intent,thinking_style,answer_format,problem_description)
Loaded 40 eval queries
Eval runs: 18 completed runs, 4 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0

In [18]:
#@title Build or resume grid plan
gs = campaign_config["grid_search"]

llm_client, llm_model = setup_llm(campaign_config)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

2026-03-10 10:34:59 INFO     [api.services.search.grid_core] Resuming grid plan gridplan_d603214cea60 (status: in_progress, 34 points)


[RESUME] Found existing grid plan: gridplan_d603214cea60
  Grid points: 34
Grid points: 34
Plan ID: gridplan_d603214cea60


In [19]:
#@title Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

2026-03-10 10:35:02 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/sessions "HTTP/1.1 200 OK"


[resume] Skipping 1/34 stored grid points, evaluating 33 remaining
  [1/34] acc=50.0% (3/6)


2026-03-10 10:35:03 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/matches "HTTP/1.1 200 OK"


        MISS 5/20  [token]  EN 10270-3-1.4310-NS-1/cold forming            -> Wire drawing, steel {RER}| wire dra 1.5s


2026-03-10 10:35:06 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/matches "HTTP/1.1 200 OK"


...

        MISS           PE                                             ERR: Client error '404 Not Found' for url 'ht


2026-03-10 10:35:09 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/matches "HTTP/1.1 404 Not Found"
2026-03-10 10:35:09 WARNING  [api.services.prompt_eval] backend_reranker_eval failed for EN 10088-2 - X5CrNi18-10+2R (1.4301)
BASF Ultrason E2010 G/s: Client error '404 Not Found' for url 'http://127.0.0.1:8000/matches'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


        MISS           EN 10088-2 - X5CrNi18-10+2R (1.4301)
BASF Ult  ERR: Client error '404 Not Found' for url 'ht


2026-03-10 10:35:10 WARNING  [api.services.search.grid_core] Grid search interrupted at point 1/34. Completed points are saved via evaluate_prompt_cached.



  [INTERRUPTED] Grid search
  Completed: 1/34 grid points
  Saved: completed points saved via evaluate_prompt_cached (content-hash dedup)
  Resume: re-run this cell -- stored points will be skipped automatically


In [ ]:
#@title Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 5. Results

In [ ]:
#@title Campaign comparison table
show_campaign_summary(campaign_rounds)

In [ ]:
#@title Per-query flip tracking (baseline vs final)
show_flip_tracking(campaign_rounds)

In [ ]:
#@title PromptState lineage chain
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)